# 💧 T16. Water Intake Coach [Health] — Agentic AI Demo Notebook

**Course Assignment**: Autonomous AI Agent Implementation with Plan-Act Loop, Function Calling, and Multi-Turn Conversation Memory.

### Core Architectural Pillars:
1. **Autonomous Plan-Act Loop**: Multi-step reasoning where the agent plans, selects tools, executes them in Python, observes structured outputs, and decides next steps.
2. **Real Function Execution**: Calls `log_water(ml)` and `get_progress()` with full validation.
3. **Persistent Python Memory**: `ConversationMemory` object maintaining state, accumulated intake, and interaction history across turns.
4. **Health Safety**: Clear non-medical disclaimer and gentle guidance without recommending dangerous over-hydration.

> *Disclaimer: This project provides simple hydration tracking and reminders for demonstration purposes. It does not provide medical advice. Individual hydration needs vary.*

In [1]:
import sys
import os
import json

# Ensure project root is in python path
sys.path.insert(0, os.path.abspath('..'))

from agent import WaterIntakeAgent
from memory import ConversationMemory
from tools import log_water, get_progress, set_daily_goal
from config import DEFAULT_DAILY_GOAL_ML, HEALTH_DISCLAIMER

def print_trace(result):
    print("=" * 75)
    print("🤖 AGENT MULTI-STEP TRACE:")
    print("-" * 75)
    for step in result['trace']:
        stype = step.get('type', 'step')
        snum = step.get('step_number', '')
        if stype == 'plan':
            print(f"[Step {snum}: PLAN] {step.get('description')}")
        elif stype == 'tool_call':
            print(f"[Step {snum}: TOOL CALL] {step.get('tool')}({json.dumps(step.get('arguments', {}))})")
        elif stype == 'tool_result':
            print(f"[Step {snum}: TOOL RESULT] {json.dumps(step.get('result', {}), indent=2)}")
        elif stype == 'observation':
            print(f"[Step {snum}: OBSERVATION] {step.get('description')}")
        elif stype == 'decision':
            print(f"[Step {snum}: DECISION] {step.get('description')}")
    print("-" * 75)
    print(f"💬 FINAL RESPONSE: {result['response']}")
    print("=" * 75 + "\n")

## 🧪 Scenario 1: Logging Water Intake

**User Input**: `"I just drank 500 ml of water."`

**Agent Workflow**:
1. **Plan**: Detect hydration intent and extract 500 ml quantity.
2. **Act 1**: Call `log_water(500)` to register consumption.
3. **Act 2**: Call `get_progress()` to refresh complete daily statistics.
4. **Observe & Decide**: Evaluate remaining intake (2000 ml left, 20% complete).
5. **Respond**: Deliver tailored status update.

In [2]:
# Initialize a fresh agent instance with empty memory
memory = ConversationMemory(daily_goal_ml=DEFAULT_DAILY_GOAL_ML)
agent = WaterIntakeAgent(memory=memory)

prompt_1 = "I just drank 500 ml of water."
print(f"👤 USER: {prompt_1}\n")
res_1 = agent.run(prompt_1)
print_trace(res_1)

## 🧪 Scenario 2: Memory & Context Persistence Across Turns

The agent must demonstrate stateful conversation memory across turns without resetting or losing prior intake amounts.

- **Turn 1 (cont.)**: User logs an additional `300 ml` (`"I drank another 300 ml."`)
- **Turn 2**: User queries total progress (`"How much have I had today?"`)
- **Expected Outcome**: Total is correctly aggregated to `800 ml` (500 ml + 300 ml).

In [3]:
# Turn 2.1: Add 300 ml to ongoing session
prompt_2a = "I drank another 300 ml."
print(f"👤 USER: {prompt_2a}\n")
res_2a = agent.run(prompt_2a)
print_trace(res_2a)

# Turn 2.2: Inquire about current cumulative progress
prompt_2b = "How much have I had today?"
print(f"👤 USER: {prompt_2b}\n")
res_2b = agent.run(prompt_2b)
print_trace(res_2b)

# Inspect direct Python memory state
print("🧠 RAW CONVERSATION MEMORY STATE:")
print(json.dumps(agent.memory.to_dict(), indent=2))

## 🧪 Scenario 3: Goal Evaluation (Approaching, Reaching, and Exceeding Goal)

Demonstrates how the agent's decision logic adapts based on boundary conditions:
1. **Approaching Goal** (e.g. 2300 / 2500 ml -> 200 ml remaining)
2. **Goal Reached** (e.g. 2500 / 2500 ml -> 100% completed)
3. **Goal Exceeded** (e.g. 2800 / 2500 ml -> gentle reminder, no over-hydration encouragement)

In [4]:
# 3.1 Approaching goal: add 1500 ml (800 + 1500 = 2300 ml / 2500 ml)
prompt_3a = "I just drank 1500 ml of water from my large jug."
print(f"👤 USER: {prompt_3a}\n")
res_3a = agent.run(prompt_3a)
print_trace(res_3a)

# 3.2 Reaching goal: add 200 ml (2300 + 200 = 2500 ml -> Goal Met)
prompt_3b = "Had another 200 ml with lunch."
print(f"👤 USER: {prompt_3b}\n")
res_3b = agent.run(prompt_3b)
print_trace(res_3b)

# 3.3 Exceeding goal: add 300 ml (2500 + 300 = 2800 ml -> Goal Exceeded)
prompt_3c = "I drank 300 ml more after working out."
print(f"👤 USER: {prompt_3c}\n")
res_3c = agent.run(prompt_3c)
print_trace(res_3c)

## 📋 Final Memory & Turn History Verification

Review all logged intake entries and turn records in persistent memory.

In [5]:
state = agent.memory.get_state()
print("📊 Final Session Summary:")
print(f"- Total Today: {state['today_intake_ml']} ml")
print(f"- Daily Goal: {state['daily_goal_ml']} ml")
print(f"- Progress: {state['progress_percent']}%")
print(f"- Goal Met: {state['goal_met']}")
print(f"- Total Logs Recorded: {state['total_logs']}")
print(f"- Total Conversation Turns: {state['total_turns']}")